# LLMOps: Observability, Caching, Routing, Cost and Deployment

Notebook for **Chapter 13** of the *genai* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/genai/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 Observability: Logs, Traces and Metrics

In [ ]:
import json, random
random.seed(1)
clock=[0.0]   # simulated clock in seconds, so the output is repeatable
def sleep_ms(ms): clock[0]+=ms/1000
PRICE_IN, PRICE_OUT = 1.00, 4.00   # made-up units per 1M tokens
spans=[]; stack=[]
class span:
    def __init__(s,name,**attrs): s.name=name; s.attrs=attrs
    def __enter__(s):
        s.t=clock[0]; s.parent=stack[-1] if stack else None; stack.append(s.name); return s
    def __exit__(s,*a):
        stack.pop(); spans.append({"span":s.name,"parent":s.parent,"ms":round((clock[0]-s.t)*1000,1),**s.attrs})
def fake_llm(prompt_tokens):
    sleep_ms(5.0); out=random.randint(150,250); return out
def handle(q):
    with span("request"):
        with span("retrieve",k=4): sleep_ms(2.5)
        with span("generate") as g:
            tin=1500; tout=fake_llm(tin)
            g.attrs.update(tokens_in=tin,tokens_out=tout,cost=round((tin*PRICE_IN+tout*PRICE_OUT)/1e6,6))
        sleep_ms(0.1)   # glue code
handle("q1")
for s in spans: print(json.dumps(s))

{"span": "retrieve", "parent": "request", "ms": 2.5, "k": 4}
{"span": "generate", "parent": "request", "ms": 5.0, "tokens_in": 1500, "tokens_out": 167, "cost": 0.002168}
{"span": "request", "parent": null, "ms": 7.6}


## 13.2 Latency Engineering: TTFT, Tokens per Second and Capacity

In [ ]:
def latency(ttft,out,tps): return ttft+out/tps
print("latency 0.6s TTFT, 300 tokens, 50 tok/s:", latency(0.6,300,50))
print("same, 150 tokens:", latency(0.6,150,50))
rate=12; L=latency(0.6,300,50); print("in flight:", round(rate*L,1))

latency 0.6s TTFT, 300 tokens, 50 tok/s: 6.6
same, 150 tokens: 3.6
in flight: 79.2


## 13.3 Caching and Routing

In [ ]:
import random
random.seed(5)
pool=[f"q{i}" for i in range(200)]; w=[1/(i+1) for i in range(200)]
reqs=random.choices(pool,weights=w,k=1000)
seen=set(); hits=0
for r in reqs:
    if r in seen: hits+=1
    seen.add(r)
print("hits:",hits,"hit rate:",hits/1000)

hits: 834 hit rate: 0.834


In [ ]:
def route(prompt):
    return "large" if len(prompt.split())>20 or "analyze" in prompt else "small"
for p in ["hi there","please analyze this report","word "*25]: print(route(p), "<-", p[:26])

small <- hi there
large <- please analyze this report
large <- word word word word word w


In [ ]:
small,large=1.0,15.0
for rr in (0.1,0.3,0.5,0.8): print("cascade escalate %.0f%%: %.1f per request"%(rr*100,small+rr*large))
print("break-even escalation:", (large-small)/large)

cascade escalate 10%: 2.5 per request
cascade escalate 30%: 5.5 per request
cascade escalate 50%: 8.5 per request
cascade escalate 80%: 13.0 per request
break-even escalation: 0.9333333333333333


## 13.4 Cost Engineering

In [ ]:
PIN,POUT=1.00,4.00
def cost(tin,tout,cached_frac=0.0,cache_disc=0.1):
    return (tin*(1-cached_frac)*PIN+tin*cached_frac*PIN*cache_disc+tout*POUT)/1e6
base=cost(2000,400); print("base per request:",base)
print("monthly:",round(200_000*base,2))
c1=cost(2000,400,cached_frac=0.75); print("prompt cache 75%:",c1,"saving %.1f%%"%((1-c1/base)*100))
c2=cost(2000,250); print("output cap 250:",c2,"saving %.1f%%"%((1-c2/base)*100))
c3=cost(1200,400); print("fewer chunks:",c3,"saving %.1f%%"%((1-c3/base)*100))
c4=cost(1200,250,cached_frac=0.5); print("combined:",c4,"saving %.1f%%"%((1-c4/base)*100))

base per request: 0.0036
monthly: 720.0
prompt cache 75%: 0.00225 saving 37.5%
output cap 250: 0.003 saving 16.7%
fewer chunks: 0.0028 saving 22.2%
combined: 0.00166 saving 53.9%


## 13.5 Deployment, Versioning and CI Evals

In [ ]:
import math, random
random.seed(11)
base=[1]*78+[0]*22; cand=[1]*84+[0]*16
def acc(x): return sum(x)/len(x)
def moe(p,n): return 1.96*math.sqrt(p*(1-p)/n)
print("baseline %.2f +-%.3f  candidate %.2f +-%.3f"%(acc(base),moe(acc(base),100),acc(cand),moe(acc(cand),100)))
THRESH_ABS=0.75; MAX_DROP=0.02
def gate(b,c):
    fails=[]
    if acc(c)<THRESH_ABS: fails.append("below absolute floor")
    if acc(c)<acc(b)-MAX_DROP: fails.append("regression vs baseline")
    return fails
print("gate good:",gate(base,cand))
bad=[1]*70+[0]*30
print("gate bad:",gate(base,bad))

baseline 0.78 +-0.081  candidate 0.84 +-0.072
gate good: []
gate bad: ['below absolute floor', 'regression vs baseline']


## 13.6 Percentiles From Raw Latency Data

In [ ]:
import math
import numpy as np
lat = [0.8, 0.9, 0.9, 1.0, 1.0, 1.1, 1.1, 1.2, 1.2, 1.3, 1.3, 1.4, 1.5, 1.6, 1.8, 2.0, 2.4, 3.1, 6.2, 12.5]
s = sorted(lat); n = len(s)
print("mean", round(sum(s) / n, 3))
for p in (50, 95, 99):
    print("p%d nearest-rank %.1f | numpy linear %.3f" % (p, s[math.ceil(p / 100 * n) - 1], np.percentile(s, p)))

mean 2.215
p50 nearest-rank 1.3 | numpy linear 1.300
p95 nearest-rank 6.2 | numpy linear 6.515
p99 nearest-rank 12.5 | numpy linear 11.303


## 13.7 Queues and Capacity

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
def simulate(lam, mu, n=200000):
    gaps = rng.exponential(1 / lam, n)      # time between arrivals
    serv = rng.exponential(1 / mu, n)       # service time of each request
    wait, total = 0.0, 0.0
    for i in range(n - 1):
        total += wait + serv[i]
        wait = max(0.0, wait + serv[i] - gaps[i + 1])
    return total / (n - 1)
for lam in (1.0, 1.6, 1.8):
    print("lambda %.1f  formula %.2f s  simulated %.2f s" % (lam, 1 / (2 - lam), simulate(lam, 2.0)))

lambda 1.0  formula 1.00 s  simulated 1.00 s
lambda 1.6  formula 2.50 s  simulated 2.48 s
lambda 1.8  formula 5.00 s  simulated 4.98 s


## 13.8 Cache Break-Even and Router Trade-Offs

**A semantic cache is not free.** It embeds every incoming question and searches the stored ones. That costs something on every request, hit or miss, and a wrong hit serves a wrong answer. So you need a break-even hit rate.

In [ ]:
c_llm, c_look = 0.0035, 0.0001
print("break-even hit rate:", round(c_look / c_llm, 4))
for h in (0.0, 0.02, 0.1, 0.3, 0.5):
    cost = c_look + (1 - h) * c_llm
    print("hit %.2f cost %.5f saving %.1f%%" % (h, cost, 100 * (1 - cost / c_llm)))
print("wrong answers per 1000 requests:", 1000 * 0.4 * 0.02)

break-even hit rate: 0.0286
hit 0.00 cost 0.00360 saving -2.9%
hit 0.02 cost 0.00353 saving -0.9%
hit 0.10 cost 0.00325 saving 7.1%
hit 0.30 cost 0.00255 saving 27.1%
hit 0.50 cost 0.00185 saving 47.1%
wrong answers per 1000 requests: 8.0


In [ ]:
easy = 0.6
quality = {"small": (0.95, 0.60), "large": (0.97, 0.90)}
cost = {"small": 1, "large": 15}
def mix(p_small_easy, p_small_hard):
    c = easy * (p_small_easy * cost["small"] + (1 - p_small_easy) * cost["large"]) + (1 - easy) * (p_small_hard * cost["small"] + (1 - p_small_hard) * cost["large"])
    q = easy * (p_small_easy * quality["small"][0] + (1 - p_small_easy) * quality["large"][0]) + (1 - easy) * (p_small_hard * quality["small"][1] + (1 - p_small_hard) * quality["large"][1])
    return c, q
for name, (a, b) in [("all large", (0, 0)), ("all small", (1, 1)), ("perfect router", (1, 0)), ("router 90% right", (0.9, 0.1)), ("router 70% right", (0.7, 0.3))]:
    c, q = mix(a, b)
    print("%-17s cost %5.2f quality %.3f" % (name, c, q))

all large         cost 15.00 quality 0.942
all small         cost  1.00 quality 0.810
perfect router    cost  6.60 quality 0.930
router 90% right  cost  6.88 quality 0.919
router 70% right  cost  7.44 quality 0.898


## 13.9 Self-Hosting Versus an API: Break-Even and GPU Sizing

An API charges per token. A rented GPU charges per hour whether it is busy or idle. The question is when the fixed cost wins. The numbers below are hypothetical (a rented GPU at 2.00 per hour, a quantised 7B-class model, 1,000 output tokens per second in aggregate across batched requests); check real prices and measure real throughput on your own model before deciding.

In [ ]:
api = 0.0035
gpu_hr, ops, hours = 2.0, 500, 720
fixed = gpu_hr * hours + ops
print("fixed monthly:", fixed, "| break-even requests:", round(fixed / api))
cap = 1000 / 250 * 3600 * hours
print("capacity at 100%:", int(cap), "| at 50%:", int(cap * 0.5))
print("self-host tokens per unit at 50%:", 1000 * 3600 / gpu_hr * 0.5, "| API output tokens per unit:", 1e6 / 4.0)
print("two GPUs, break-even:", round((2 * gpu_hr * hours + ops) / api))

fixed monthly: 1940.0 | break-even requests: 554286
capacity at 100%: 10368000 | at 50%: 5184000
self-host tokens per unit at 50%: 900000.0 | API output tokens per unit: 250000.0
two GPUs, break-even: 965714


## Chapter recap

## Practice set

**1.** (Easy) A request has 1,000 input and 200 output tokens at 1.00 and 4.00 per million. Cost?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/genai/ch13.html